# Central Differential Privacy Neural Network Training

Dataset: **IPBlock Fraud Detection** (Amazon Fraud Dataset Benchmark)

This notebook corresponds to **Section 6 "Private-model workflow (Central
Differential Privacy)"** of the paper, applied to the fraud detection task
on the IPBlock dataset.

It trains a feedforward neural network under DP-SGD using TensorFlow
Privacy and runs the grid search over sample size, batch size, noise
multiplier, learning rate, and clipping norm that produces Figures 6, 8,
10, 12, 14, and 16 of the paper.

> **Environment**: run this notebook in the `.cdp` virtual environment
> (TensorFlow 2.3.0 + tensorflow-privacy 0.5.1; see `requirements/cdp.txt`).
> The `.ldp` environment uses Keras 3 and will not work here.

> **Data prerequisites**: the three IPBlock CSVs must exist in
> `../data/raw/`. To produce them, run once from the `.fdb` environment:
> `python -m src.fdb_export.export_data`.

In [1]:
import os
import random
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix,
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.metrics import AUC
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow_privacy.privacy.optimizers.dp_optimizer_keras import DPKerasSGDOptimizer
from tensorflow_privacy.privacy.analysis import compute_dp_sgd_privacy

import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm

# Paths
ROOT = Path.cwd().resolve()                 # FraudDetection/notebooks
DATA_RAW = ROOT.parent / "data" / "raw"
FIG_DIR = ROOT.parent / "figures"
RES_DIR = ROOT.parent / "results"
FIG_DIR.mkdir(parents=True, exist_ok=True)
RES_DIR.mkdir(parents=True, exist_ok=True)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# Experiment configuration
TARGET = "EVENT_LABEL"

# Network architecture (fixed across the grid search)
HIDDEN_UNITS = 64
HIDDEN_LAYERS = 2
DROPOUT_RATE = 0.2
EPOCHS = 50

# Number of independent grid-search repetitions
N_REPEATS = 10


In [2]:
# Load IPBlock CSVs and align test labels with test features
train_path = DATA_RAW / "ipblock_train.csv"
test_feat_path = DATA_RAW / "ipblock_test_features.csv"
test_labels_path = DATA_RAW / "ipblock_test_labels.csv"

df_train = pd.read_csv(train_path)
df_test_features = pd.read_csv(test_feat_path)
df_test_labels = pd.read_csv(test_labels_path)

if "EVENT_ID" in df_test_features.columns and "EVENT_ID" in df_test_labels.columns:
    df_test = df_test_features.merge(
        df_test_labels[["EVENT_ID", TARGET]],
        on="EVENT_ID",
        how="left",
    )
else:
    raise ValueError("EVENT_ID is not present in both test tables.")

print("Train shape:", df_train.shape)
print("Test shape :", df_test.shape)


Train shape: (172000, 8)
Test shape : (43000, 7)


In [3]:
# Feature engineering (matches FraudDetection/05 verbatim)
def engineer_features(df):
    df = df.copy()

    # IP -> first two octets
    df["ip"] = df["ip"].astype(str)
    df["ip_1"] = df["ip"].str.split(".").str[0]
    df["ip_2"] = df["ip"].str.split(".").str[1]

    # Timestamp -> hour, day of week, month, weekend indicator
    dt = pd.to_datetime(df["EVENT_TIMESTAMP"])
    df["event_hour"] = dt.dt.hour
    df["event_dow"] = dt.dt.dayofweek
    df["event_month"] = dt.dt.month
    df["event_is_weekend"] = df["event_dow"].isin([5, 6]).astype(int)

    feature_cols = [
        "ip_1", "ip_2",
        "ENTITY_TYPE",
        "event_hour", "event_dow", "event_month", "event_is_weekend",
    ]

    X = df[feature_cols].copy()
    y = df[TARGET].astype(int).copy()
    return X, y, feature_cols


X_train_raw, y_train, feature_cols = engineer_features(df_train)
X_test_raw,  y_test,  _           = engineer_features(df_test)

print("X_train_raw:", X_train_raw.shape, "y_train:", y_train.shape)
y_train.value_counts()


X_train_raw: (172000, 7) y_train: (172000,)


0    159997
1     12003
Name: EVENT_LABEL, dtype: int64

In [4]:
# One-hot encoding aligned across train and test
def one_hot_encode_train_test(X_train, X_test, categorical_cols):
    X_train_oh = pd.get_dummies(
        X_train, columns=categorical_cols, drop_first=True, dtype="float32",
    )
    X_test_oh = pd.get_dummies(
        X_test, columns=categorical_cols, drop_first=True, dtype="float32",
    )
    X_test_oh = X_test_oh.reindex(columns=X_train_oh.columns, fill_value=0.0)
    return X_train_oh, X_test_oh


categorical_cols = feature_cols
X_train_oh, X_test_oh = one_hot_encode_train_test(X_train_raw, X_test_raw, categorical_cols)

print("X_train_oh:", X_train_oh.shape)
print("X_test_oh :", X_test_oh.shape)

INPUT_SIZE = X_train_oh.shape[1]
X_train_full = X_train_oh.values
X_test_full  = X_test_oh.values
y_train_full = y_train.values
y_test_full  = y_test.values


X_train_oh: (172000, 517)
X_test_oh : (43000, 517)


In [5]:
# Initial 1:1 undersampling of the training set
def compute_undersample_indices(y, negative_to_positive_ratio=1.0, random_state=42):
    rng = np.random.RandomState(random_state)
    y_arr = np.asarray(y)

    idx_pos = np.where(y_arr == 1)[0]
    idx_neg = np.where(y_arr == 0)[0]
    n_pos = len(idx_pos)
    n_neg = len(idx_neg)

    n_neg_desired = min(n_neg, int(n_pos * negative_to_positive_ratio))
    idx_neg_sample = rng.choice(idx_neg, size=n_neg_desired, replace=False)
    idx_keep = np.concatenate([idx_pos, idx_neg_sample])
    idx_keep.sort()
    return idx_keep


train_idx_keep = compute_undersample_indices(
    y_train_full,
    negative_to_positive_ratio=1.0,
    random_state=SEED,
)

X_train_bal = X_train_full[train_idx_keep]
y_train_bal = y_train_full[train_idx_keep]

print("Train class distribution (original):")
print(pd.Series(y_train_full).value_counts())
print("\nTrain class distribution (balanced):")
print(pd.Series(y_train_bal).value_counts())

N_TRAIN_BAL = len(X_train_bal)


Train class distribution (original):
0    159997
1     12003
dtype: int64

Train class distribution (balanced):
1    12003
0    12003
dtype: int64


## Helper functions

In [6]:
# Compute the DP-SGD privacy budget for the given training configuration
def compute_privacy_budget(n, batch_size, noise_multiplier, epochs, delta=1e-5):
    try:
        eps, _ = compute_dp_sgd_privacy.compute_dp_sgd_privacy(
            n=n, batch_size=batch_size,
            noise_multiplier=noise_multiplier,
            epochs=epochs, delta=delta,
        )
        return eps
    except Exception as e:
        print("Error computing epsilon:", e)
        return float("inf")


In [7]:
# Feedforward network with an optional DP-SGD optimizer (paper §6.1)
def create_model(
    input_size, hidden_units, hidden_layers, dropout_rate,
    learning_rate, num_microbatches, l2_norm_clip,
    noise_multiplier, use_dp,
):
    model = Sequential()
    model.add(Dense(hidden_units, activation="relu", input_shape=(input_size,)))
    for _ in range(hidden_layers - 1):
        model.add(Dense(hidden_units, activation="relu"))
        model.add(Dropout(dropout_rate))
    model.add(Dense(1, activation="sigmoid"))

    if use_dp:
        optimizer = DPKerasSGDOptimizer(
            l2_norm_clip=l2_norm_clip,
            noise_multiplier=noise_multiplier,
            num_microbatches=num_microbatches,
            learning_rate=learning_rate,
        )
    else:
        optimizer = Adam(learning_rate=learning_rate)

    model.compile(
        optimizer=optimizer,
        loss="binary_crossentropy",
        metrics=[AUC(name="auc"), "accuracy"],
    )
    return model


In [8]:
# Train one model on the given fold and return its test-set predictions
def train_model(
    X_train, y_train, X_test, y_test,
    batch_size, epochs, learning_rate,
    use_dp, noise_multiplier, l2_norm_clip,
    use_early_stopping=True,
):
    model = create_model(
        input_size=INPUT_SIZE,
        hidden_units=HIDDEN_UNITS,
        hidden_layers=HIDDEN_LAYERS,
        dropout_rate=DROPOUT_RATE,
        learning_rate=learning_rate,
        num_microbatches=batch_size,
        l2_norm_clip=l2_norm_clip,
        noise_multiplier=noise_multiplier,
        use_dp=use_dp,
    )

    callbacks = []
    if use_early_stopping:
        callbacks.append(EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True))

    model.fit(
        X_train, y_train,
        batch_size=batch_size, epochs=epochs,
        validation_split=0.2,
        callbacks=callbacks,
        verbose=0,
    )

    y_prob = model.predict(X_test, batch_size=batch_size).flatten()
    y_pred = (y_prob > 0.5).astype(int)
    return y_prob, y_pred


In [9]:
# Compute the metric panel used in the paper from test-set predictions
def evaluate_model(y_true, y_pred, y_prob):
    cm = confusion_matrix(y_true, y_pred)
    return {
        "ROC AUC":       roc_auc_score(y_true, y_prob),
        "Accuracy":      accuracy_score(y_true, y_pred),
        "Precision":     precision_score(y_true, y_pred, zero_division=0),
        "Recall":        recall_score(y_true, y_pred, zero_division=0),
        "F1 Score":      f1_score(y_true, y_pred, zero_division=0),
        "Type I Error":  cm[0][1] / cm[0].sum() if cm[0].sum() > 0 else 0.0,
        "Type II Error": cm[1][0] / cm[1].sum() if cm[1].sum() > 0 else 0.0,
    }


In [11]:
# One full experiment: train, evaluate, and tag with its configuration
def run_experiment(
    X_train_data, y_train_data,
    batch_size, learning_rate, noise_multiplier, l2_norm_clip,
    use_dp=True, use_early_stopping=True,
):
    n = len(X_train_data)

    if use_dp and noise_multiplier > 0.0 and l2_norm_clip > 0.0:
        eps = compute_privacy_budget(
            n=n, batch_size=batch_size,
            noise_multiplier=noise_multiplier,
            epochs=EPOCHS, delta=1e-5,
        )
    else:
        eps = np.inf

    y_prob, y_pred = train_model(
        X_train_data, y_train_data, X_test_full, y_test_full,
        batch_size=batch_size, epochs=EPOCHS, learning_rate=learning_rate,
        use_dp=use_dp, noise_multiplier=noise_multiplier, l2_norm_clip=l2_norm_clip,
        use_early_stopping=use_early_stopping,
    )
    results = evaluate_model(y_test_full, y_pred, y_prob)
    results.update({
        "Epsilon":          eps,
        "Batch Size":       batch_size,
        "Noise Multiplier": noise_multiplier,
        "Learning Rate":    learning_rate,
        "Clipping Norm":    l2_norm_clip,
        # Sample Ratio is relative to the balanced training set
        "Sample Ratio":     len(X_train_data) / N_TRAIN_BAL,
        "DP Enabled":       use_dp,
    })
    return results


In [12]:
# Full grid search: non-DP baseline and DP-SGD sweep over five hyperparameters,
# replicated across four sub-sampling ratios.
def grid_search_experiments(use_early_stopping=True):
    batch_sizes       = [16, 32, 64, 128]
    noise_multipliers = [0.8, 1.1, 1.5, 2.0]
    learning_rates    = [0.001, 0.003, 0.005]
    clip_norms        = [0.5, 1.0, 2.0]
    sample_ratios     = [1.0, 0.5, 0.1, 0.05]

    results = []
    for ratio in sample_ratios:
        n_samples = int(N_TRAIN_BAL * ratio)
        idx = np.random.choice(N_TRAIN_BAL, n_samples, replace=False)
        X_sample = X_train_bal[idx]
        y_sample = y_train_bal[idx]

        # Non-DP baseline: varies batch size and learning rate only
        for bs, lr in product(batch_sizes, learning_rates):
            results.append(run_experiment(
                X_sample, y_sample,
                batch_size=bs, learning_rate=lr,
                noise_multiplier=0.0, l2_norm_clip=0.0,
                use_dp=False, use_early_stopping=use_early_stopping,
            ))

        # DP-SGD: varies batch size, noise multiplier, learning rate, clipping norm
        for bs, noise, lr, clip in product(
            batch_sizes, noise_multipliers, learning_rates, clip_norms,
        ):
            results.append(run_experiment(
                X_sample, y_sample,
                batch_size=bs, learning_rate=lr,
                noise_multiplier=noise, l2_norm_clip=clip,
                use_dp=True, use_early_stopping=use_early_stopping,
            ))

    return pd.DataFrame(results)


## Run grid search

In [13]:
# Repeat the entire grid search N_REPEATS times to absorb training stochasticity
all_runs = []
for run_id in range(N_REPEATS):
    print(f"--- Grid search run {run_id + 1}/{N_REPEATS} ---")
    df_run = grid_search_experiments(use_early_stopping=True)
    df_run["Run"] = run_id + 1
    all_runs.append(df_run)


--- Grid search run 1/10 ---
DP-SGD with sampling rate = 0.0667% and noise_multiplier = 0.8 iterated over 75019 steps satisfies differential privacy with eps = 2.17 and delta = 1e-05.
The optimal RDP order is 8.0.
DP-SGD with sampling rate = 0.0667% and noise_multiplier = 0.8 iterated over 75019 steps satisfies differential privacy with eps = 2.17 and delta = 1e-05.
The optimal RDP order is 8.0.
DP-SGD with sampling rate = 0.0667% and noise_multiplier = 0.8 iterated over 75019 steps satisfies differential privacy with eps = 2.17 and delta = 1e-05.
The optimal RDP order is 8.0.
DP-SGD with sampling rate = 0.0667% and noise_multiplier = 0.8 iterated over 75019 steps satisfies differential privacy with eps = 2.17 and delta = 1e-05.
The optimal RDP order is 8.0.
DP-SGD with sampling rate = 0.0667% and noise_multiplier = 0.8 iterated over 75019 steps satisfies differential privacy with eps = 2.17 and delta = 1e-05.
The optimal RDP order is 8.0.
DP-SGD with sampling rate = 0.0667% and noise_

In [14]:
# Combine runs and aggregate to mean/min/max per configuration
df_all = pd.concat(all_runs, ignore_index=True)
df_all.round(3).to_csv(RES_DIR / "cdp_ipblock_all_runs.csv", index=False)

metrics = [
    "ROC AUC", "Accuracy", "Precision", "Recall",
    "F1 Score", "Type I Error", "Type II Error", "Epsilon",
]
group_cols = [
    "Batch Size", "Noise Multiplier", "Learning Rate",
    "Clipping Norm", "Sample Ratio", "DP Enabled",
]

agg_results = (
    df_all.groupby(group_cols)[metrics]
          .agg(["mean", "min", "max"])
          .reset_index()
)
agg_results.columns = [" ".join(col).strip() for col in agg_results.columns.values]
agg_results.round(3).to_csv(RES_DIR / "cdp_ipblock_aggregated_results.csv", index=False)


## Plot results

In [16]:
# Plotting setup: style, output subfolders, reload aggregated results
plt.style.use("seaborn")

for sub in ("cdp1", "cdp2", "cdp3", "cdp4", "cdp5"):
    (FIG_DIR / sub).mkdir(parents=True, exist_ok=True)

df_results = pd.read_csv(RES_DIR / "cdp_ipblock_aggregated_results.csv")
non_dp_data = df_results[df_results["DP Enabled"] == False]
dp_data     = df_results[df_results["DP Enabled"] == True]

# Grid axes used by the plots below
sample_ratios     = [1.0, 0.5, 0.1, 0.05]
batch_sizes       = [16, 32, 64, 128]
learning_rates    = [0.001, 0.003, 0.005]
noise_multipliers = [0.8, 1.1, 1.5, 2.0]
clip_norms        = [0.5, 1.0, 2.0]

VMIN, VMAX = 0.5, 0.84


In [17]:
# Plot 1: non-DP ROC AUC heatmaps per sample ratio (paper Figure 6)
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
axes = axes.ravel()
fig.suptitle("ROC AUC for non-DP experiments (fraud detection)",
             fontsize=18, fontweight="bold")

for i, ratio in enumerate(sample_ratios):
    subset = non_dp_data[non_dp_data["Sample Ratio"] == ratio]
    pivot = subset.pivot(index="Batch Size", columns="Learning Rate", values="ROC AUC mean")
    pivot = pivot.reindex(index=batch_sizes, columns=learning_rates, fill_value=np.nan)

    sns.heatmap(
        pivot, annot=True, fmt=".3f", cmap="YlGnBu", ax=axes[i],
        cbar_kws={"label": "ROC AUC"}, annot_kws={"size": 13},
        vmin=VMIN, vmax=VMAX,
    )
    axes[i].set_title(f"Sample Ratio: {ratio}", fontsize=16)
    axes[i].set_xlabel("Learning Rate", fontsize=14)
    axes[i].set_ylabel("Batch Size", fontsize=14)
    axes[i].tick_params(axis="both", labelsize=12)
    cbar = axes[i].collections[0].colorbar
    cbar.ax.tick_params(labelsize=12)
    cbar.set_label("ROC AUC", fontsize=14)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig(FIG_DIR / "cdp1" / "non_dp_roc_auc_heatmaps_ipblock.png")
plt.close()


In [18]:
# Plot 2: DP epsilon heatmaps per sample ratio (paper Figure 8)
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
axes = axes.ravel()
fig.suptitle("Epsilon for DP experiments by sample ratio (fraud detection)",
             fontsize=18, fontweight="bold")

for i, ratio in enumerate(sample_ratios):
    subset = dp_data[dp_data["Sample Ratio"] == ratio]
    pivot = subset.pivot_table(
        index="Batch Size", columns="Noise Multiplier",
        values="Epsilon mean", aggfunc="mean",
    )
    pivot = pivot.reindex(index=batch_sizes, columns=noise_multipliers, fill_value=np.nan)

    sns.heatmap(
        pivot, annot=True, fmt=".3f", cmap="YlGnBu", ax=axes[i],
        cbar_kws={"label": "Epsilon"}, annot_kws={"size": 13},
    )
    axes[i].set_title(f"Sample Ratio: {ratio}", fontsize=16)
    axes[i].set_xlabel("Noise Multiplier", fontsize=14)
    axes[i].set_ylabel("Batch Size", fontsize=14)
    axes[i].tick_params(axis="both", labelsize=12)
    cbar = axes[i].collections[0].colorbar
    cbar.ax.tick_params(labelsize=12)
    cbar.set_label("Epsilon", fontsize=14)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig(FIG_DIR / "cdp2" / "dp_epsilon_heatmaps_ipblock.png")
plt.close()


In [19]:
# Plot 3: DP ROC AUC heatmaps at batch_size=16 across four noise multipliers
# (paper Figure 10). For each fixed (batch, noise), the panel sweeps over
# sample ratios; each cell varies clipping norm vs. learning rate.
configurations = [{"batch_size": 16, "noise_multiplier": nm}
                  for nm in noise_multipliers]

for config in configurations:
    bs = config["batch_size"]
    nm = config["noise_multiplier"]
    config_data = dp_data[(dp_data["Batch Size"] == bs) & (dp_data["Noise Multiplier"] == nm)]

    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    axes = axes.ravel()
    fig.suptitle(
        f"ROC AUC for DP (fraud detection), Batch Size={bs}, Noise Multiplier={nm}",
        fontsize=18, fontweight="bold",
    )

    for i, ratio in enumerate(sample_ratios):
        subset = config_data[config_data["Sample Ratio"] == ratio]
        epsilon = subset["Epsilon mean"].mean() if not subset.empty else np.nan
        epsilon_str = f"{epsilon:.3f}" if not np.isnan(epsilon) else "N/A"

        pivot = subset.pivot(index="Clipping Norm", columns="Learning Rate", values="ROC AUC mean")
        pivot = pivot.reindex(index=clip_norms, columns=learning_rates, fill_value=np.nan)

        sns.heatmap(
            pivot, annot=True, fmt=".3f", cmap="YlGnBu", ax=axes[i],
            cbar_kws={"label": "ROC AUC"}, vmin=VMIN, vmax=VMAX, annot_kws={"size": 13},
        )
        axes[i].set_title(f"Sample Ratio: {ratio}, Epsilon: {epsilon_str}", fontsize=16)
        axes[i].set_xlabel("Learning Rate", fontsize=14)
        axes[i].set_ylabel("Clipping Norm", fontsize=14)
        axes[i].tick_params(axis="both", labelsize=12)
        cbar = axes[i].collections[0].colorbar
        cbar.ax.tick_params(labelsize=12)
        cbar.set_label("ROC AUC", fontsize=14)

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(FIG_DIR / "cdp3" / f"dp_roc_auc_bs_{bs}_nm_{nm}_ipblock.png")
    plt.close()


In [20]:
# Plot 4: For each sample ratio, ROC AUC heatmaps at batch_size=16 across the
# four noise multipliers (paper Figures 12, 14)
noise_configurations = [{"batch_size": 16, "noise_multiplier": nm}
                        for nm in noise_multipliers]

for ratio in sample_ratios:
    ratio_data = dp_data[dp_data["Sample Ratio"] == ratio]

    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    axes = axes.ravel()
    fig.suptitle(
        f"ROC AUC for DP (fraud detection), Sample Ratio={ratio}",
        fontsize=18, fontweight="bold",
    )

    for i, cfg in enumerate(noise_configurations):
        bs, nm = cfg["batch_size"], cfg["noise_multiplier"]
        subset = ratio_data[(ratio_data["Batch Size"] == bs) & (ratio_data["Noise Multiplier"] == nm)]
        epsilon = subset["Epsilon mean"].mean() if not subset.empty else np.nan
        epsilon_str = f"{epsilon:.3f}" if not np.isnan(epsilon) else "N/A"

        pivot = subset.pivot(index="Clipping Norm", columns="Learning Rate", values="ROC AUC mean")
        pivot = pivot.reindex(index=clip_norms, columns=learning_rates, fill_value=np.nan)

        sns.heatmap(
            pivot, annot=True, fmt=".3f", cmap="YlGnBu", ax=axes[i],
            cbar_kws={"label": "ROC AUC"}, vmin=VMIN, vmax=VMAX, annot_kws={"size": 13},
        )
        axes[i].set_title(f"Batch Size: {bs}, Noise: {nm}\nEpsilon: {epsilon_str}", fontsize=16)
        axes[i].set_xlabel("Learning Rate", fontsize=14)
        axes[i].set_ylabel("Clipping Norm", fontsize=14)
        axes[i].tick_params(axis="both", labelsize=12)
        cbar = axes[i].collections[0].colorbar
        cbar.ax.tick_params(labelsize=12)
        cbar.set_label("ROC AUC", fontsize=14)

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(FIG_DIR / "cdp4" / f"dp_roc_auc_bs_{bs}_nm_{nm}_sr_{ratio}_ipblock.png")
    plt.close()


In [21]:
# Plot 5: For each sample ratio, ROC AUC heatmaps at noise_multiplier=1.5 across
# the four batch sizes (paper Figure 16)
batch_configurations = [{"batch_size": bs, "noise_multiplier": 1.5}
                        for bs in batch_sizes]

for ratio in sample_ratios:
    ratio_data = dp_data[dp_data["Sample Ratio"] == ratio]

    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    axes = axes.ravel()
    fig.suptitle(
        f"ROC AUC for DP (fraud detection), Sample Ratio={ratio}",
        fontsize=18, fontweight="bold",
    )

    for i, cfg in enumerate(batch_configurations):
        bs, nm = cfg["batch_size"], cfg["noise_multiplier"]
        subset = ratio_data[(ratio_data["Batch Size"] == bs) & (ratio_data["Noise Multiplier"] == nm)]
        epsilon = subset["Epsilon mean"].mean() if not subset.empty else np.nan
        epsilon_str = f"{epsilon:.3f}" if not np.isnan(epsilon) else "N/A"

        pivot = subset.pivot(index="Clipping Norm", columns="Learning Rate", values="ROC AUC mean")
        pivot = pivot.reindex(index=clip_norms, columns=learning_rates, fill_value=np.nan)

        sns.heatmap(
            pivot, annot=True, fmt=".3f", cmap="YlGnBu", ax=axes[i],
            cbar_kws={"label": "ROC AUC"}, vmin=VMIN, vmax=VMAX, annot_kws={"size": 13},
        )
        axes[i].set_title(f"Batch Size: {bs}, Noise: {nm}\nEpsilon: {epsilon_str}", fontsize=16)
        axes[i].set_xlabel("Learning Rate", fontsize=14)
        axes[i].set_ylabel("Clipping Norm", fontsize=14)
        axes[i].tick_params(axis="both", labelsize=12)
        cbar = axes[i].collections[0].colorbar
        cbar.ax.tick_params(labelsize=12)
        cbar.set_label("ROC AUC", fontsize=14)

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(FIG_DIR / "cdp5" / f"dp_roc_auc_bs_{bs}_nm_{nm}_sr_{ratio}_ipblock.png")
    plt.close()


## ANOVA

In [22]:
# Non-DP ANOVA: 2-way partial interactions over batch size, sample ratio, learning rate
anova_non_dp_data = non_dp_data[
    ["ROC AUC mean", "Batch Size", "Learning Rate", "Sample Ratio"]
].copy().astype({
    "Batch Size":    "category",
    "Learning Rate": "category",
    "Sample Ratio":  "category",
})

formula_non_dp = (
    'Q("ROC AUC mean") ~ '
    'C(Q("Batch Size")) * C(Q("Sample Ratio")) + '
    'C(Q("Batch Size")) * C(Q("Learning Rate")) + '
    'C(Q("Sample Ratio")) * C(Q("Learning Rate"))'
)

model_non_dp = smf.ols(formula=formula_non_dp, data=anova_non_dp_data).fit()
anova_lm(model_non_dp, typ=2)


,sum_sq,df,F,PR(>F)
"C(Q(""Batch Size""))",0.000027,3.0,0.418133,7.421476e-01
"C(Q(""Sample Ratio""))",0.222790,3.0,3442.247318,5.148301e-25
"C(Q(""Learning Rate""))",0.000117,2.0,2.715451,9.318620e-02
"C(Q(""Batch Size"")):C(Q(""Sample Ratio""))",0.000562,9.0,2.893670,2.629992e-02
"C(Q(""Batch Size"")):C(Q(""Learning Rate""))",0.000095,6.0,0.733906,6.288549e-01
"C(Q(""Sample Ratio"")):C(Q(""Learning Rate""))",0.000642,6.0,4.960944,3.690837e-03
Residual,0.000388,18.0,NaN,NaN


In [23]:
# DP ANOVA: main effects over all five DP-SGD hyperparameters
anova_dp_data = dp_data[
    ["ROC AUC mean", "Batch Size", "Noise Multiplier",
     "Clipping Norm", "Learning Rate", "Sample Ratio"]
].copy().astype({
    "Batch Size":       "category",
    "Noise Multiplier": "category",
    "Clipping Norm":    "category",
    "Learning Rate":    "category",
    "Sample Ratio":     "category",
})

formula_dp = (
    'Q("ROC AUC mean") ~ '
    'C(Q("Batch Size")) + C(Q("Noise Multiplier")) + '
    'C(Q("Clipping Norm")) + C(Q("Learning Rate")) + C(Q("Sample Ratio"))'
)

model_dp = smf.ols(formula=formula_dp, data=anova_dp_data).fit()
anova_lm(model_dp, typ=2)


,sum_sq,df,F,PR(>F)
"C(Q(""Batch Size""))",0.975078,3.0,167.035278,2.102967e-77
"C(Q(""Noise Multiplier""))",0.000177,3.0,0.030236,9.929202e-01
"C(Q(""Clipping Norm""))",0.000013,2.0,0.003320,9.966856e-01
"C(Q(""Learning Rate""))",0.626081,2.0,160.875858,5.716087e-56
"C(Q(""Sample Ratio""))",2.878998,3.0,493.185448,6.088691e-157
Residual,1.093569,562.0,NaN,NaN
